# Divar Fatemi Rental Anomaly Scanner
این نوت‌بوک کل pipeline را از GitHub اجرا می‌کند: crawl → normalize → anomaly/duplicate → optional Jev → report.

> امتیاز `suspicion_score` اثبات تقلب نیست؛ یک ابزار triage برای بازبینی انسانی است.

In [ ]:
!rm -rf /content/divar_scanner
!git clone https://github.com/prestigegitserp/divar_scanner.git /content/divar_scanner
%cd /content/divar_scanner
!python -m pip install -q -e .


## تنظیمات اجرا
برای تست اول 100–200 آگهی مناسب است. Jev اختیاری است و کلید در سورس ذخیره نمی‌شود.

In [ ]:
import os, getpass, yaml
from pathlib import Path

MAX_LISTINGS = 200  #@param {type:"integer"}
USE_JEV = False  #@param {type:"boolean"}

cfg = yaml.safe_load(Path('config/fatemi.yaml').read_text(encoding='utf-8'))
cfg['crawl']['max_listings'] = int(MAX_LISTINGS)
cfg['semantic']['enabled'] = bool(USE_JEV)
runtime_cfg = Path('/content/fatemi_colab.yaml')
runtime_cfg.write_text(yaml.safe_dump(cfg, allow_unicode=True, sort_keys=False), encoding='utf-8')

if USE_JEV:
    key = getpass.getpass('JEV_API_KEY (hidden): ').strip()
    if key:
        os.environ['JEV_API_KEY'] = key
    else:
        print('No key entered; disabling Jev for this run.')
        cfg['semantic']['enabled'] = False
        runtime_cfg.write_text(yaml.safe_dump(cfg, allow_unicode=True, sort_keys=False), encoding='utf-8')

print(runtime_cfg.read_text(encoding='utf-8'))


In [ ]:
from divar_scanner.pipeline import run_pipeline

df, meta = run_pipeline('/content/fatemi_colab.yaml', crawl=True)
print(meta['counts'])
print(meta['outputs'])


In [ ]:
from IPython.display import display

cols = [
    'suspicion_score','risk_band','title','neighborhood','area_m2','rooms',
    'deposit_toman','rent_monthly_toman','market_anomaly_score',
    'data_quality_score','duplicate_similarity','semantic_score',
    'jev_classification','flag_reasons','url'
]
display(df[[c for c in cols if c in df.columns]].head(40))


In [ ]:
# دانلود فایل‌های خروجی از Colab
from google.colab import files
for label, path in meta.get('outputs', {}).items():
    print(label, path)
# مثال: files.download(meta['outputs']['review_csv'])


## پیشنهاد برای فاز بعد
بعد از بازبینی دستی حداقل چندصد نمونه، یک ستون label انسانی اضافه کنید و وزن‌ها/thresholdها یا یک LightGBM ranker را روی همان labels کالیبره کنید.